In [ ]:
#Imports first
import os
import boto3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

#Ensure local 'images' directory exists for exporting plots
os.makedirs("images", exist_ok=True)

#Set global Seaborn styling
sns.set_theme(style="whitegrid")

NETID = "np767"
bucket_name = f"dsan6000-{NETID.lower()}"
s3_prefix = "wikipedia-hourly"

print(f"Target Bucket: s3://{bucket_name}/{s3_prefix}/")

In [ ]:
#List and Read Parquet Data directly from S3
s3_client = boto3.client("s3")

#List all objects in your personal S3 bucket under 'wikipedia-hourly/'
response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=s3_prefix)

#Filter for .parquet files
parquet_keys = [
    obj["Key"] for obj in response.get("Contents", []) 
    if obj["Key"].endswith(".parquet")
]

#Read each .parquet file directly from S3 using Pandas
dfs = []
for key in sorted(parquet_keys):
    s3_uri = f"s3://{bucket_name}/{key}"
    print(f"Reading directly from S3: {s3_uri}")
    df_hour = pd.read_parquet(s3_uri)
    dfs.append(df_hour)

#Concatenate all 24 hourly DataFrames into one full DataFrame
full_df = pd.concat(dfs, ignore_index=True)
print(f"\nTotal events loaded across all 24 hours: {len(full_df):,}")

In [ ]:
#Data Aggregation

#Parse datetime column and floor to hour
full_df["datetime"] = pd.to_datetime(full_df["datetime"])
full_df["hour"] = full_df["datetime"].dt.floor("h")

#Aggregate 1: Total event counts per hour
hourly_total = full_df.groupby("hour").size().reset_index(name="event_count")

#Aggregate 2: Hourly event counts grouped by 'type'
hourly_by_type = full_df.groupby(["hour", "type"]).size().reset_index(name="event_count")

In [ ]:
#Plot 1 total events per hour

plt.figure(figsize=(12, 6))
ax1 = sns.lineplot(
    data=hourly_total, 
    x="hour", 
    y="event_count", 
    marker="o", 
    linewidth=2.5
)

plt.title("Total Wikipedia Events per Hour", fontsize=14, fontweight="bold")
plt.xlabel("Hour of Day", fontsize=12)
plt.ylabel("Total Event Count", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()

# Export plot in both .png and .svg formats
plt.savefig("images/hourly-events.png", dpi=300)
plt.savefig("images/hourly-events.svg")
plt.show()

In [ ]:
#Plot 2 events by type

plt.figure(figsize=(12, 6))
ax2 = sns.lineplot(
    data=hourly_by_type, 
    x="hour", 
    y="event_count", 
    hue="type", 
    marker="o", 
    linewidth=2
)

plt.title("Wikipedia Event Counts per Hour by Event Type", fontsize=14, fontweight="bold")
plt.xlabel("Hour of Day", fontsize=12)
plt.ylabel("Event Count", fontsize=12)
plt.xticks(rotation=45)
plt.legend(title="Event Type", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

# Export plot in both .png and .svg formats
plt.savefig("images/events-by-type.png", dpi=300)
plt.savefig("images/events-by-type.svg")
plt.show()